# Imports & Functions

In [1]:
# %load_ext autoreload
# %autoreload 2

In [11]:
import pandas as pd
import numpy as np
import glob
import os
from march_madness.config import RAW_DATA_DIR, INTERIM_DATA_DIR

In [3]:
def create_id(df, columns, new_column_name, separator=""):
    df[new_column_name] = df[columns].astype(str).agg(separator.join, axis=1)
    return df

# Load Data

In [4]:
# Path to the folder
folder_path = RAW_DATA_DIR / 'march-machine-learning-mania-2026'

# Find all CSV files starting with "M"
pattern = os.path.join(folder_path, "M*.csv")
csv_files = glob.glob(pattern)

In [5]:
if not csv_files:
    print("No CSV files starting with 'M' found in the specified folder.")
else:
    dataframes = {}
    for file in csv_files:
        filename = os.path.basename(file)
        df = pd.read_csv(file)
        dataframes[filename] = df
        print(f"Loaded '{filename}': {df.shape[0]} rows, {df.shape[1]} columns")

Loaded 'MConferenceTourneyGames.csv': 6793 rows, 5 columns
Loaded 'MGameCities.csv': 90684 rows, 6 columns
Loaded 'MMasseyOrdinals.csv': 5761702 rows, 5 columns
Loaded 'MNCAATourneyCompactResults.csv': 2585 rows, 8 columns
Loaded 'MNCAATourneyDetailedResults.csv': 1449 rows, 34 columns
Loaded 'MNCAATourneySeedRoundSlots.csv': 776 rows, 5 columns
Loaded 'MNCAATourneySeeds.csv': 2626 rows, 3 columns
Loaded 'MNCAATourneySlots.csv': 2586 rows, 4 columns
Loaded 'MRegularSeasonCompactResults.csv': 196823 rows, 8 columns
Loaded 'MRegularSeasonDetailedResults.csv': 122775 rows, 34 columns
Loaded 'MSeasons.csv': 42 rows, 6 columns
Loaded 'MSecondaryTourneyCompactResults.csv': 1865 rows, 9 columns
Loaded 'MSecondaryTourneyTeams.csv': 1895 rows, 3 columns
Loaded 'MTeamCoaches.csv': 13898 rows, 5 columns
Loaded 'MTeamConferences.csv': 13753 rows, 3 columns
Loaded 'MTeams.csv': 381 rows, 4 columns
Loaded 'MTeamSpellings.csv': 1178 rows, 2 columns


# Clean Data

In [6]:
df_reg_detail_results = dataframes.get('MRegularSeasonDetailedResults.csv').copy()
df_reg_compact_results = dataframes.get('MRegularSeasonCompactResults.csv').copy()
df_tn_detail_results = dataframes.get('MNCAATourneyDetailedResults.csv').copy()
df_seeds = dataframes.get('MNCAATourneySeeds.csv').copy()

In [7]:
df_reg_detail_results = create_id(df_reg_detail_results, ['Season', 'DayNum', 'WTeamID', 'LTeamID'], 'id', separator="_")
df_reg_compact_results = create_id(df_reg_compact_results, ['Season', 'DayNum', 'WTeamID', 'LTeamID'], 'id', separator="_")
df_tn_detail_results = create_id(df_tn_detail_results, ['Season', 'DayNum', 'WTeamID', 'LTeamID'], 'id', separator="_")
df_seeds = create_id(df_seeds, ['Season', 'Seed', 'TeamID'], 'id', separator="_")

# Base summary stats

In [8]:
df_wins = df_reg_compact_results[['Season', 'WTeamID', 'WScore']].assign(wins=1).set_axis(['season', 'team_id', 'points_in_wins', 'wins'], axis=1)
df_wins_agg = df_wins.groupby(['season', 'team_id']).sum().reset_index()

df_losses = df_reg_compact_results[['Season', 'LTeamID', 'LScore']].assign(losses=1).set_axis(['season', 'team_id', 'points_in_losses', 'losses'], axis=1)
df_losses_agg = df_losses.groupby(['season', 'team_id']).sum().reset_index()

df_results_agg = df_wins_agg.merge(df_losses_agg, on = ['season', 'team_id'], how = 'outer')
df_results_agg['points'] = df_results_agg['points_in_wins'] + df_results_agg['points_in_losses']
df_results_agg = df_results_agg.fillna(0)


# Export clean data

In [ ]:
df_reg_detail_results.to_parquet(INTERIM_DATA_DIR / 'df_reg_detail_results.parquet', engine='pyarrow', index=False)
df_tn_detail_results.to_parquet(INTERIM_DATA_DIR / 'df_tn_detail_results.parquet', engine='pyarrow', index=False)
df_seeds.to_parquet(INTERIM_DATA_DIR / 'df_seeds.parquet', engine='pyarrow', index=False)